In [1]:
!pip install pandas openpyxl xlrd

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
'''PDF TO TEXT'''

#PDF TO TEXT CONVERTER
import pandas as pd  # #PDF TO TEXT PO1
import re
import pdfplumber

def extract_data_from_pdf(pdf_path):
    """
    Extract data from PDF file and return structured data
    """
    data = []
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Extract text from all pages
            full_text = ""
            for page in pdf.pages:
                full_text += page.extract_text() + "\n"
            
            print("Extracted text from PDF:")
            print("=" * 50)
            print(full_text)
            print("=" * 50)
            
            # Process the extracted text
            lines = full_text.split('\n')
            current_item = {}
            
            for line in lines:
                line = line.strip()
                if not line:
                    continue
                    
                # Extract SrNo (looking for lines starting with numbers)
                sr_no_match = re.match(r'^\s*(\d+)\s*', line)
                if sr_no_match:
                    if current_item and 'SrNo' in current_item:  # Save previous item if exists
                        data.append(current_item)
                    current_item = {'SrNo': sr_no_match.group(1), 'OrderQty': 2}
                    
                # Extract Article code (patterns like 9-DD106-YG-25, 9-DD106-YG-7)
                article_match = re.search(r'([A-Z0-9\-]+YG\-?\d*\.?\d+)', line)
                if article_match and 'ArticleCode' not in current_item:
                    current_item['ArticleCode'] = article_match.group(1)
                    
                # Extract StyleCode from Your reference (text after RSBR2074-01, RSBR2074-03, etc.)
                style_match = re.search(r'RSBR2074\-(\d+)\s+([A-Z0-9]+)', line)
                if style_match:
                    current_item['StyleCode'] = style_match.group(2)
                    
                # Extract ItemSize from description (looking for numbers like 0.25, 0.07)
                # Multiple patterns to catch different formats
                size_patterns = [
                    r'pendant\s+(\d+\.\d+)',  # pendant 0.25
                    r'(\d+\.\d+)\s*ct',       # 0.25 ct
                    r'YG[-\s]*(\d+\.\d+)',    # YG-0.25 or YG 0.25
                ]
                
                for pattern in size_patterns:
                    size_match = re.search(pattern, line, re.IGNORECASE)
                    if size_match and 'ItemSize' not in current_item:
                        current_item['ItemSize'] = size_match.group(1)
                        break
            
            # Add the last item
            if current_item and 'SrNo' in current_item:
                data.append(current_item)
                
    except Exception as e:
        print(f"Error reading PDF file: {e}")
        return []
    
    return data

def create_excel_dataframe(data):
    """
    Create DataFrame with specified columns
    """
    if not data:
        print("No data extracted from PDF")
        return pd.DataFrame()
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Reorder columns as requested
    columns_order = ['SrNo', 'StyleCode', 'ItemSize', 'OrderQty']
    
    # Only include columns that exist in the data
    final_columns = [col for col in columns_order if col in df.columns]
    
    # Add missing columns with empty values
    for col in columns_order:
        if col not in df.columns:
            df[col] = ""
    
    return df[columns_order]

# Main execution
if __name__ == "__main__":
    # Specify your PDF file path here
   
    pdf_file_path = r'C:\Users\Admin\Desktop\OPS_Tool_10032026\SHEFI_NEW_PO\PDPO#29626.pdf'  # Change this to your actual PDF path
    
    # Extract data from PDF
    print(f"Reading PDF from: {pdf_file_path}")
    extracted_data = extract_data_from_pdf(pdf_file_path)
    
    if extracted_data:
        # Create DataFrame
        df = create_excel_dataframe(extracted_data)
        
        # Display the results
        print("\nExtracted Data:")
        print("=" * 50)
        print(df)
        
        # Save to Excel file
        output_file = 'extracted_data.xlsx'
        df.to_excel(output_file, index=False)
        print(f"\nData successfully saved to '{output_file}'")
        
        # Display basic statistics
        print(f"\nExtraction Summary:")
        print(f"Total records extracted: {len(df)}")
        print(f"Columns: {list(df.columns)}")
        
    else:
        print("No data was extracted from the PDF file.")
    
    # Display the DataFrame in notebook
    df

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


Reading PDF from: C:\Users\Admin\Desktop\OPS_Tool_10032026\SHEFI_NEW_PO\PDPO#29626.pdf
Extracted text from PDF:
oT
redrO
PRESTIGE DIAMONDS PVT. LTD SHEFI DIAMONDS, INC.
BDB BKC ANDHERI E 10 WEST 46TH STREET, 15TH FLOOR
MUMBAI MH INDIA NEW YORK NY 10036 USA
RAVINA/ANKUSH/LOKESH
oT
pihS
SHEFI DIAMONDS, INC. Purchase Order
10 WEST 46TH STREET, 15TH FLOOR
NEW YORK NY 10036 USA
(800) 76-SHEFI (212) 391-1482 Fax: (212) 719-2892
WWW.SHEFIDIAMONDS.COM
INFO@SHEFIDIAMONDS.COM
Order #: 29626
Page #: 1 of 2
P.O. #: STOCK Date: 3/27/2026 Due Date: 3/27/2026 Cancel Date: 3/27/2026
Reference: STOCK Vendor #:P533 Phone #: Ship Via: BRINKS
# Memo # Item # Vendor Item # Job Bag # Description Size Quantity Weight Unit Cost Amount
LGD Anniversary:
1 LGD244910E 14KW Shared Prong 6.5 1 0.0000 Q $1.00 $1.00
11 Rd. 1/10 Ctw Lab
Grown Diamonds
LGD Anniversary:
2 LGD244910D 14KY Shared Prong 11 6.5 1 0.0000 Q $1.00 $1.00
Rd. 1/10 Ctw Lab
Grown Diamonds
LGD Anniversary:
3 LGD244925E 14KW Shared Prong 6.5 1 0.000

In [6]:
'''EXTRACT COMPLETE PO DATA WITH ALL COLUMNS - CORRECTED'''

import pandas as pd
import re
from datetime import datetime

def extract_complete_po_data(pdf_text):
    """
    Extract all PO data according to specified column requirements
    """
    lines = pdf_text.split('\n')
    data = []
    
    # Extract header information for each page
    header_info = {}
    
    # First pass: extract all header information from both pages
    for line in lines:
        line = line.strip()
        
        # Order #
        order_match = re.search(r'Order #:\s*(\d+)', line)
        if order_match and 'Order#' not in header_info:
            header_info['Order#'] = order_match.group(1)
        
        # Page # (capture both pages)
        page_match = re.search(r'Page #:\s*(\d+\s+of\s+\d+)', line)
        if page_match:
            header_info['Page#'] = page_match.group(1)
        
        # PO #
        po_match = re.search(r'P\.O\. #:\s*(\w+)', line)
        if po_match and 'PO#' not in header_info:
            header_info['PO#'] = po_match.group(1)
        
        # Date
        date_match = re.search(r'Date:\s*(\d+/\d+/\d+)', line)
        if date_match and 'Date' not in header_info:
            date_str = date_match.group(1)
            try:
                date_obj = datetime.strptime(date_str, '%m/%d/%Y')
                header_info['Date'] = date_obj.strftime('%d-%b-%y')
            except:
                header_info['Date'] = date_str
        
        # Due Date
        due_match = re.search(r'Due Date:\s*(\d+/\d+/\d+)', line)
        if due_match and 'Due Date' not in header_info:
            due_str = due_match.group(1)
            try:
                due_obj = datetime.strptime(due_str, '%m/%d/%Y')
                header_info['Due Date'] = due_obj.strftime('%d-%b-%y')
            except:
                header_info['Due Date'] = due_str
        
        # Cancel Date
        cancel_match = re.search(r'Cancel Date:\s*(\d+/\d+/\d+)', line)
        if cancel_match and 'Cancel Date' not in header_info:
            cancel_str = cancel_match.group(1)
            try:
                cancel_obj = datetime.strptime(cancel_str, '%m/%d/%Y')
                header_info['Cancel Date'] = cancel_obj.strftime('%d-%b-%y')
            except:
                header_info['Cancel Date'] = cancel_str
        
        # Reference
        ref_match = re.search(r'Reference:\s*(\w+)', line)
        if ref_match and 'Ref' not in header_info:
            header_info['Ref'] = ref_match.group(1)
        
        # Vendor #
        vendor_match = re.search(r'Vendor #:\s*(\w+)', line)
        if vendor_match and 'Vendor#' not in header_info:
            header_info['Vendor#'] = vendor_match.group(1)
        
        # Ship Via
        ship_match = re.search(r'Ship Via:\s*(\w+)', line)
        if ship_match and 'Ship Via' not in header_info:
            header_info['Ship Via'] = ship_match.group(1)
    
    # Second pass: extract item details
    current_item = {}
    item_counter = 0
    description_lines = []
    capture_next_line = False
    
    for i, line in enumerate(lines):
        line = line.strip()
        
        # Look for LGD Anniversary lines (indicate new item coming)
        if 'LGD Anniversary:' in line:
            capture_next_line = True
            continue
        
        # Look for item lines (starting with numbers)
        if capture_next_line:
            item_match = re.match(r'^(\d+)\s+([A-Z0-9]+)\s+(.+)', line)
            if item_match:
                # Save previous item if exists
                if current_item:
                    # Add description lines collected
                    if description_lines:
                        current_item['Description'] = ' '.join(description_lines)
                    data.append(current_item)
                
                # Start new item
                item_counter += 1
                sr_no = item_match.group(1)
                item_no = item_match.group(2)
                rest_of_line = item_match.group(3)
                
                current_item = {
                    'Customer': 'SHEFI DIAMONDS, INC',
                    'Order#': header_info.get('Order#', ''),
                    'Page#': header_info.get('Page#', ''),
                    'PO#': header_info.get('PO#', ''),
                    'Date': header_info.get('Date', ''),
                    'Due Date': header_info.get('Due Date', ''),
                    'Cancel Date': header_info.get('Cancel Date', ''),
                    'Ref': header_info.get('Ref', ''),
                    'Vendor#': header_info.get('Vendor#', ''),
                    'Ship Via': header_info.get('Ship Via', ''),
                    '#': item_counter,
                    'Memo #': '',
                    'Item #': item_no,
                    'Vendor Item #': '',
                    'Description': 'LGD Anniversary',
                    'Size': '',
                    'Quantity': ''
                }
                
                # Parse the rest of the line for additional details
                parts = rest_of_line.split()
                
                # Look for Vendor Item # (usually 4th word like RG0004161E)
                if len(parts) >= 3 and re.match(r'^[A-Z0-9]{8,}$', parts[2]):
                    current_item['Vendor Item #'] = parts[2]
                
                # Extract Size (number after 14KW/14KY and before the next number)
                size_match = re.search(r'14K[WDY]\s+.*?(\d+\.?\d*)', rest_of_line)
                if size_match:
                    current_item['Size'] = size_match.group(1)
                
                # Extract Quantity (single digit, usually 1)
                qty_match = re.search(r'\b(\d+)\b(?![A-Z0-9])', rest_of_line)
                if qty_match:
                    current_item['Quantity'] = qty_match.group(1)
                
                # Reset description lines
                description_lines = ['LGD Anniversary']
                capture_next_line = False
        
        # Collect description lines for current item
        elif current_item and ('14KW' in line or '14KY' in line or 'Shared Prong' in line):
            description_lines.append(line)
        elif current_item and ('Rd.' in line or 'Ctw' in line or 'Lab' in line or 'Grown Diamonds' in line):
            description_lines.append(line)
    
    # Add the last item
    if current_item:
        if description_lines:
            current_item['Description'] = ' '.join(description_lines)
        data.append(current_item)
    
    return data

# Get the PDF text
pdf_file_path = r'C:\Users\Admin\Desktop\OPS_Tool_10032026\SHEFI_NEW_PO\PDPO#29626.pdf'

import pdfplumber
with pdfplumber.open(pdf_file_path) as pdf:
    full_text = ""
    for page in pdf.pages:
        full_text += page.extract_text() + "\n"

# Extract complete PO data
complete_data = extract_complete_po_data(full_text)

# Create DataFrame
if complete_data:
    # Define column order
    columns_order = [
        'Customer', 'Order#', 'Page#', 'PO#', 'Date', 'Due Date', 'Cancel Date',
        'Ref', 'Vendor#', 'Ship Via', '#', 'Memo #', 'Item #', 'Vendor Item #',
        'Description', 'Size', 'Quantity'
    ]
    
    df_complete = pd.DataFrame(complete_data)
    
    # Ensure all columns exist and are in correct order
    for col in columns_order:
        if col not in df_complete.columns:
            df_complete[col] = ''
    
    df_complete = df_complete[columns_order]
    
    print("Complete PO Data:")
    print("=" * 80)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 50)
    print(df_complete)
    
    # Save to Excel file
    output_file = 'COMPLETE_PO_DATA.xlsx'
    df_complete.to_excel(output_file, index=False)
    print(f"\nComplete PO data saved to '{output_file}'")
    
    # Display summary
    print(f"\nExtraction Summary:")
    print(f"Total items: {len(df_complete)}")
    print(f"Columns: {list(df_complete.columns)}")
    
    # Show first few items in detail
    print(f"\nFirst 3 items detailed view:")
    print(df_complete.head(3).to_string(index=False))
    
else:
    print("No data extracted")

# Display the complete DataFrame
df_complete

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


Complete PO Data:
               Customer Order#   Page#    PO#       Date   Due Date  \
0   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
1   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
2   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
3   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
4   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
5   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
6   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
7   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
8   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
9   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
10  SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
11  SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   

   Cancel Date    Ref Vendor# Ship Via   # Memo #      Ite

,Customer,Order#,Page#,PO#,Date,Due Date,Cancel Date,Ref,Vendor#,Ship Via,#,Memo #,Item #,Vendor Item #,Description,Size,Quantity
0,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,1,,LGD244910E,,LGD Anniversary 11 Rd. 1/10 Ctw Lab Grown Diam...,6.5,6
1,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,2,,LGD244910D,,LGD Anniversary Rd. 1/10 Ctw Lab Grown Diamonds,11,11
2,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,3,,LGD244925E,,LGD Anniversary 11 Rd. 1/4 Ctw Lab Grown Diamonds,6.5,6
3,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,4,,LGD244925D,,LGD Anniversary Rd. 1/4 Ctw Lab Grown Diamonds,11,11
4,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,5,,LGD244933E,,LGD Anniversary 11 Rd. 1/3 Ctw Lab Grown Diamonds,6.5,6
5,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,6,,LGD244933D,,LGD Anniversary Rd. 1/3 Ctw Lab Grown Diamonds,11,11
6,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,7,,LGD244950D,,LGD Anniversary Rd. 1/2 Ctw Lab Grown Diamonds,11,11
7,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,8,,LGD244950E,,LGD Anniversary 11 Rd. 1/2 Ctw Lab Grown Diamonds,6.5,6
8,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,9,,LGD244975E,,LGD Anniversary 11 Rd. 3/4 Ctw Lab Grown Diamonds,6.5,6
9,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,10,,LGD244975D,,LGD Anniversary Rd. 3/4 Ctw Lab Grown Diamonds,11,11


In [ ]:
'''CORRECTED PO DATA EXTRACTION - FIXING ALL ISSUES'''

import pandas as pd
import re
from datetime import datetime

def extract_complete_po_data_corrected(pdf_text):
    """
    Extract all PO data according to specified column requirements - CORRECTED VERSION
    """
    lines = pdf_text.split('\n')
    data = []
    
    # Extract header information
    header_info = {}
    
    # Extract header information from both pages
    for line in lines:
        line = line.strip()
        
        # Order #
        order_match = re.search(r'Order #:\s*(\d+)', line)
        if order_match:
            header_info['Order#'] = order_match.group(1)
        
        # Page # (capture all page info)
        page_match = re.search(r'Page #:\s*(\d+\s+of\s+\d+)', line)
        if page_match:
            header_info['Page#'] = page_match.group(1)
        
        # PO #
        po_match = re.search(r'P\.O\. #:\s*(\w+)', line)
        if po_match:
            header_info['PO#'] = po_match.group(1)
        
        # Date
        date_match = re.search(r'Date:\s*(\d+/\d+/\d+)', line)
        if date_match:
            date_str = date_match.group(1)
            try:
                date_obj = datetime.strptime(date_str, '%m/%d/%Y')
                header_info['Date'] = date_obj.strftime('%d-%b-%y')
            except:
                header_info['Date'] = date_str
        
        # Due Date
        due_match = re.search(r'Due Date:\s*(\d+/\d+/\d+)', line)
        if due_match:
            due_str = due_match.group(1)
            try:
                due_obj = datetime.strptime(due_str, '%m/%d/%Y')
                header_info['Due Date'] = due_obj.strftime('%d-%b-%y')
            except:
                header_info['Due Date'] = due_str
        
        # Cancel Date
        cancel_match = re.search(r'Cancel Date:\s*(\d+/\d+/\d+)', line)
        if cancel_match:
            cancel_str = cancel_match.group(1)
            try:
                cancel_obj = datetime.strptime(cancel_str, '%m/%d/%Y')
                header_info['Cancel Date'] = cancel_obj.strftime('%d-%b-%y')
            except:
                header_info['Cancel Date'] = cancel_str
        
        # Reference
        ref_match = re.search(r'Reference:\s*(\w+)', line)
        if ref_match:
            header_info['Ref'] = ref_match.group(1)
        
        # Vendor #
        vendor_match = re.search(r'Vendor #:\s*(\w+)', line)
        if vendor_match:
            header_info['Vendor#'] = vendor_match.group(1)
        
        # Ship Via
        ship_match = re.search(r'Ship Via:\s*(\w+)', line)
        if ship_match:
            header_info['Ship Via'] = ship_match.group(1)
    
    # Extract item details
    current_item = {}
    item_counter = 0
    capture_next_line = False
    
    for i, line in enumerate(lines):
        line = line.strip()
        
        # Look for LGD Anniversary lines (indicate new item coming)
        if 'LGD Anniversary:' in line:
            capture_next_line = True
            continue
        
        # Look for item lines (starting with numbers)
        if capture_next_line:
            item_match = re.match(r'^(\d+)\s+([A-Z0-9]+)\s+(.+)', line)
            if item_match:
                # Save previous item if exists
                if current_item:
                    data.append(current_item)
                
                # Start new item
                item_counter += 1
                sr_no = item_match.group(1)
                item_no = item_match.group(2)
                rest_of_line = item_match.group(3)
                
                # Split the rest of line into words for precise extraction
                words = rest_of_line.split()
                
                # Initialize item with header info
                current_item = {
                    'Customer': 'SHEFI DIAMONDS, INC',
                    'Order#': header_info.get('Order#', ''),
                    'Page#': header_info.get('Page#', ''),
                    'PO#': header_info.get('PO#', ''),
                    'Date': header_info.get('Date', ''),
                    'Due Date': header_info.get('Due Date', ''),
                    'Cancel Date': header_info.get('Cancel Date', ''),
                    'Ref': header_info.get('Ref', ''),
                    'Vendor#': header_info.get('Vendor#', ''),
                    'Ship Via': header_info.get('Ship Via', ''),
                    '#': item_counter,
                    'Memo #': '',
                    'Item #': item_no,
                    'Vendor Item #': '',
                    'Description': '',
                    'Size': '',
                    'Quantity': ''
                }
                
                # Extract Vendor Item # (4th word if it matches pattern)
                if len(words) >= 3 and re.match(r'^[A-Z0-9]{8,}$', words[2]):
                    current_item['Vendor Item #'] = words[2]
                
                # Extract metal type and description parts
                description_parts = []
                
                # Look for 14KW/14KY pattern and what follows
                metal_pattern = re.search(r'(14K[WDY]\s+Shared\s+Prong)', rest_of_line)
                if metal_pattern:
                    description_parts.append(metal_pattern.group(1))
                
                # Extract Size (6th word - after metal description)
                if len(words) >= 5:
                    # Size is typically the 6th word (index 5)
                    size_match = re.match(r'(\d+\.?\d*)', words[5])
                    if size_match:
                        current_item['Size'] = size_match.group(1)
                
                # Extract Quantity (7th word)
                if len(words) >= 6:
                    qty_match = re.match(r'(\d+)', words[6])
                    if qty_match:
                        current_item['Quantity'] = qty_match.group(1)
                
                # Build description starting with LGD Anniversary
                current_item['Description'] = 'LGD Anniversary'
                if metal_pattern:
                    current_item['Description'] += ' ' + metal_pattern.group(1)
                
                capture_next_line = False
        
        # Collect additional description lines for current item
        elif current_item and ('Rd.' in line or 'Ctw' in line or 'Lab' in line or 'Grown Diamonds' in line):
            # Add diamond details to description
            if current_item['Description']:
                current_item['Description'] += ' ' + line
            else:
                current_item['Description'] = line
    
    # Add the last item
    if current_item:
        data.append(current_item)
    
    return data

# Get the PDF text from the first cell's extraction
pdf_file_path = r'C:\Users\Admin\Desktop\OPS_Tool_10032026\SHEFI_NEW_PO\PDPO#29626.pdf'

import pdfplumber
with pdfplumber.open(pdf_file_path) as pdf:
    full_text = ""
    for page in pdf.pages:
        full_text += page.extract_text() + "\n"

# Extract complete PO data with corrections
complete_data = extract_complete_po_data_corrected(full_text)

# Create DataFrame
if complete_data:
    # Define column order
    columns_order = [
        'Customer', 'Order#', 'Page#', 'PO#', 'Date', 'Due Date', 'Cancel Date',
        'Ref', 'Vendor#', 'Ship Via', '#', 'Memo #', 'Item #', 'Vendor Item #',
        'Description', 'Size', 'Quantity'
    ]
    
    df_complete = pd.DataFrame(complete_data)
    
    # Ensure all columns exist and are in correct order
    for col in columns_order:
        if col not in df_complete.columns:
            df_complete[col] = ''
    
    df_complete = df_complete[columns_order]
    
    print("CORRECTED PO Data:")
    print("=" * 80)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 80)
    print(df_complete)
    
    # Save to Excel file
    output_file = 'CORRECTED_PO_DATA.xlsx'
    df_complete.to_excel(output_file, index=False)
    print(f"\nCorrected PO data saved to '{output_file}'")
    
    # Display summary
    print(f"\nExtraction Summary:")
    print(f"Total items: {len(df_complete)}")
    print(f"Columns: {list(df_complete.columns)}")
    
    # Show detailed view of first few items
    print(f"\nFirst 5 items detailed view:")
    for i, row in df_complete.head(5).iterrows():
        print(f"\nItem {i+1}:")
        print(f"  Item #: {row['Item #']}")
        print(f"  Vendor Item #: {row['Vendor Item #']}")
        print(f"  Description: {row['Description']}")
        print(f"  Size: {row['Size']}")
        print(f"  Quantity: {row['Quantity']}")
    
else:
    print("No data extracted")

# Display the complete DataFrame
df_complete

In [ ]:
# Placeholder cell 5

In [ ]:
# Placeholder cell 6

In [8]:
'''FINAL CORRECTED PO DATA EXTRACTION - ALL ISSUES FIXED'''

import pandas as pd
import re
from datetime import datetime

def extract_complete_po_data_final(pdf_text):
    """
    Extract all PO data according to specified column requirements - FINAL CORRECTED VERSION
    """
    lines = pdf_text.split('\n')
    data = []
    
    # Extract header information
    header_info = {}
    
    # Extract header information from both pages
    for line in lines:
        line = line.strip()
        
        # Order #
        order_match = re.search(r'Order #:\s*(\d+)', line)
        if order_match:
            header_info['Order#'] = order_match.group(1)
        
        # Page # (capture all page info)
        page_match = re.search(r'Page #:\s*(\d+\s+of\s+\d+)', line)
        if page_match:
            header_info['Page#'] = page_match.group(1)
        
        # PO #
        po_match = re.search(r'P\.O\. #:\s*(\w+)', line)
        if po_match:
            header_info['PO#'] = po_match.group(1)
        
        # Date
        date_match = re.search(r'Date:\s*(\d+/\d+/\d+)', line)
        if date_match:
            date_str = date_match.group(1)
            try:
                date_obj = datetime.strptime(date_str, '%m/%d/%Y')
                header_info['Date'] = date_obj.strftime('%d-%b-%y')
            except:
                header_info['Date'] = date_str
        
        # Due Date
        due_match = re.search(r'Due Date:\s*(\d+/\d+/\d+)', line)
        if due_match:
            due_str = due_match.group(1)
            try:
                due_obj = datetime.strptime(due_str, '%m/%d/%Y')
                header_info['Due Date'] = due_obj.strftime('%d-%b-%y')
            except:
                header_info['Due Date'] = due_str
        
        # Cancel Date
        cancel_match = re.search(r'Cancel Date:\s*(\d+/\d+/\d+)', line)
        if cancel_match:
            cancel_str = cancel_match.group(1)
            try:
                cancel_obj = datetime.strptime(cancel_str, '%m/%d/%Y')
                header_info['Cancel Date'] = cancel_obj.strftime('%d-%b-%y')
            except:
                header_info['Cancel Date'] = cancel_str
        
        # Reference
        ref_match = re.search(r'Reference:\s*(\w+)', line)
        if ref_match:
            header_info['Ref'] = ref_match.group(1)
        
        # Vendor #
        vendor_match = re.search(r'Vendor #:\s*(\w+)', line)
        if vendor_match:
            header_info['Vendor#'] = vendor_match.group(1)
        
        # Ship Via
        ship_match = re.search(r'Ship Via:\s*(\w+)', line)
        if ship_match:
            header_info['Ship Via'] = ship_match.group(1)
    
    # Extract item details
    current_item = {}
    item_counter = 0
    capture_next_line = False
    
    for i, line in enumerate(lines):
        line = line.strip()
        
        # Look for LGD Anniversary lines (indicate new item coming)
        if 'LGD Anniversary:' in line:
            capture_next_line = True
            continue
        
        # Look for item lines (starting with numbers)
        if capture_next_line:
            item_match = re.match(r'^(\d+)\s+([A-Z0-9]+)\s+(.+)', line)
            if item_match:
                # Save previous item if exists
                if current_item:
                    data.append(current_item)
                
                # Start new item
                item_counter += 1
                sr_no = item_match.group(1)
                item_no = item_match.group(2)
                rest_of_line = item_match.group(3)
                
                # Split the rest of line into words for precise extraction
                words = rest_of_line.split()
                
                # Initialize item with header info
                current_item = {
                    'Customer': 'SHEFI DIAMONDS, INC',
                    'Order#': header_info.get('Order#', ''),
                    'Page#': header_info.get('Page#', ''),
                    'PO#': header_info.get('PO#', ''),
                    'Date': header_info.get('Date', ''),
                    'Due Date': header_info.get('Due Date', ''),
                    'Cancel Date': header_info.get('Cancel Date', ''),
                    'Ref': header_info.get('Ref', ''),
                    'Vendor#': header_info.get('Vendor#', ''),
                    'Ship Via': header_info.get('Ship Via', ''),
                    '#': item_counter,
                    'Memo #': '',
                    'Item #': item_no,
                    'Vendor Item #': '',
                    'Description': '',
                    'Size': '',
                    'Quantity': ''
                }
                
                # Extract Vendor Item # (4th word if it matches pattern - FIXED)
                if len(words) >= 3:
                    # Check if 3rd word (index 2) is a vendor item code
                    if re.match(r'^[A-Z0-9]{8,}$', words[2]):
                        current_item['Vendor Item #'] = words[2]
                
                # Extract metal type for description
                metal_pattern = re.search(r'(14K[WDY]\s+Shared\s+Prong)', rest_of_line)
                
                # Extract Size (6th word - after "14KW Shared Prong") - FIXED
                if len(words) >= 5:
                    # Size is the 6th word (index 5) - should be 6.5 or 11
                    size_match = re.match(r'(\d+\.?\d*)', words[5])
                    if size_match:
                        current_item['Size'] = size_match.group(1)
                
                # Extract Quantity (7th word) - FIXED
                if len(words) >= 6:
                    # Quantity is the 7th word (index 6) - should be 1
                    qty_match = re.match(r'(\d+)', words[6])
                    if qty_match:
                        current_item['Quantity'] = qty_match.group(1)
                
                # Build description starting with LGD Anniversary + metal info
                current_item['Description'] = 'LGD Anniversary'
                if metal_pattern:
                    current_item['Description'] += ' ' + metal_pattern.group(1)
                
                capture_next_line = False
        
        # Collect additional description lines for current item
        elif current_item and ('Rd.' in line or 'Ctw' in line or 'Lab' in line or 'Grown Diamonds' in line):
            # Add diamond details to description
            current_item['Description'] += ' ' + line
    
    # Add the last item
    if current_item:
        data.append(current_item)
    
    return data

# Get the PDF text from the first cell's extraction
pdf_file_path = r'C:\Users\Admin\Desktop\OPS_Tool_10032026\SHEFI_NEW_PO\PDPO#29626.pdf'

import pdfplumber
with pdfplumber.open(pdf_file_path) as pdf:
    full_text = ""
    for page in pdf.pages:
        full_text += page.extract_text() + "\n"

# Extract complete PO data with final corrections
complete_data = extract_complete_po_data_final(full_text)

# Create DataFrame
if complete_data:
    # Define column order
    columns_order = [
        'Customer', 'Order#', 'Page#', 'PO#', 'Date', 'Due Date', 'Cancel Date',
        'Ref', 'Vendor#', 'Ship Via', '#', 'Memo #', 'Item #', 'Vendor Item #',
        'Description', 'Size', 'Quantity'
    ]
    
    df_complete = pd.DataFrame(complete_data)
    
    # Ensure all columns exist and are in correct order
    for col in columns_order:
        if col not in df_complete.columns:
            df_complete[col] = ''
    
    df_complete = df_complete[columns_order]
    
    print("FINAL CORRECTED PO Data:")
    print("=" * 80)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 80)
    print(df_complete)
    
    # Save to Excel file
    output_file = 'FINAL_CORRECTED_PO_DATA.xlsx'
    df_complete.to_excel(output_file, index=False)
    print(f"\nFinal corrected PO data saved to '{output_file}'")
    
    # Display summary
    print(f"\nExtraction Summary:")
    print(f"Total items: {len(df_complete)}")
    print(f"Columns: {list(df_complete.columns)}")
    
    # Show detailed view of first few items with verification
    print(f"\nFirst 5 items detailed view (VERIFICATION):")
    for i, row in df_complete.head(5).iterrows():
        print(f"\nItem {i+1}:")
        print(f"  Item #: {row['Item #']}")
        print(f"  Vendor Item #: '{row['Vendor Item #']}'")
        print(f"  Description: {row['Description']}")
        print(f"  Size: '{row['Size']}'")
        print(f"  Quantity: '{row['Quantity']}'")
    
    # Verify specific issues were fixed
    print(f"\n" + "="*50)
    print("VERIFICATION OF FIXES:")
    print("="*50)
    
    # Check Vendor Item # extraction
    vendor_items = df_complete[df_complete['Vendor Item #'] != '']['Item #'].tolist()
    print(f"Items with Vendor Item # extracted: {vendor_items}")
    
    # Check Size extraction
    sizes = df_complete['Size'].tolist()
    print(f"Sizes extracted: {sizes}")
    
    # Check Quantity extraction
    quantities = df_complete['Quantity'].tolist()
    print(f"Quantities extracted: {quantities}")
    
    # Check Description includes metal info
    descriptions = df_complete['Description'].tolist()
    metal_in_desc = [desc for desc in descriptions if '14KW' in desc or '14KY' in desc]
    print(f"Items with metal info in description: {len(metal_in_desc)}")
    
else:
    print("No data extracted")

# Display the complete DataFrame
df_complete

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


FINAL CORRECTED PO Data:
               Customer Order#   Page#    PO#       Date   Due Date  \
0   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
1   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
2   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
3   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
4   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
5   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
6   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
7   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
8   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
9   SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
10  SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   
11  SHEFI DIAMONDS, INC  29626  2 of 2  STOCK  27-Mar-26  27-Mar-26   

   Cancel Date    Ref Vendor# Ship Via   # Memo #  

,Customer,Order#,Page#,PO#,Date,Due Date,Cancel Date,Ref,Vendor#,Ship Via,#,Memo #,Item #,Vendor Item #,Description,Size,Quantity
0,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,1,,LGD244910E,,LGD Anniversary 14KW Shared Prong 11 Rd. 1/10 Ctw Lab Grown Diamonds,0.0000,
1,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,2,,LGD244910D,,LGD Anniversary 14KY Shared Prong Rd. 1/10 Ctw Lab Grown Diamonds,1,0
2,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,3,,LGD244925E,,LGD Anniversary 14KW Shared Prong 11 Rd. 1/4 Ctw Lab Grown Diamonds,0.0000,
3,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,4,,LGD244925D,,LGD Anniversary 14KY Shared Prong Rd. 1/4 Ctw Lab Grown Diamonds,1,0
4,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,5,,LGD244933E,,LGD Anniversary 14KW Shared Prong 11 Rd. 1/3 Ctw Lab Grown Diamonds,1,0
5,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,6,,LGD244933D,,LGD Anniversary 14KY Shared Prong Rd. 1/3 Ctw Lab Grown Diamonds,6.5,1
6,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,7,,LGD244950D,,LGD Anniversary 14KY Shared Prong Rd. 1/2 Ctw Lab Grown Diamonds,6.5,1
7,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,8,,LGD244950E,,LGD Anniversary 14KW Shared Prong 11 Rd. 1/2 Ctw Lab Grown Diamonds,1,0
8,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,9,,LGD244975E,,LGD Anniversary 14KW Shared Prong 11 Rd. 3/4 Ctw Lab Grown Diamonds,1,0
9,"SHEFI DIAMONDS, INC",29626,2 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,10,,LGD244975D,,LGD Anniversary 14KY Shared Prong Rd. 3/4 Ctw Lab Grown Diamonds,6.5,1


In [7]:
''' the text extracted from the above code cell to excel mapping'''

#FOR PDF
# Data processing for PO #6393 - Create DataFrame with all required columns
import pandas as pd
import re

def process_po_6393_data():
    """
    Process the extracted text from PO #6393 and create DataFrame with specified columns
    """
    
    # The extracted text from the PDF
    extracted_text = """oT
redrO
PRESTIGE DIAMONDS PVT. LTD SHEFI DIAMONDS, INC.
BDB BKC ANDHERI E 10 WEST 46TH STREET, 15TH FLOOR
MUMBAI MH INDIA NEW YORK NY 10036 USA
RAVINA/ANKUSH/LOKESH
oT
pihS
SHEFI DIAMONDS, INC. Purchase Order
10 WEST 46TH STREET, 15TH FLOOR
NEW YORK NY 10036 USA
(800) 76-SHEFI (212) 391-1482 Fax: (212) 719-2892
WWW.SHEFIDIAMONDS.COM
INFO@SHEFIDIAMONDS.COM
Order #: 29626
Page #: 1 of 2
P.O. #: STOCK Date: 3/27/2026 Due Date: 3/27/2026 Cancel Date: 3/27/2026
Reference: STOCK Vendor #:P533 Phone #: Ship Via: BRINKS
# Memo # Item # Vendor Item # Job Bag # Description Size Quantity Weight Unit Cost Amount
LGD Anniversary:
1 LGD244910E 14KW Shared Prong 6.5 1 0.0000 Q $1.00 $1.00
11 Rd. 1/10 Ctw Lab
Grown Diamonds
LGD Anniversary:
2 LGD244910D 14KY Shared Prong 11 6.5 1 0.0000 Q $1.00 $1.00
Rd. 1/10 Ctw Lab
Grown Diamonds
LGD Anniversary:
3 LGD244925E 14KW Shared Prong 6.5 1 0.0000 Q $1.00 $1.00
11 Rd. 1/4 Ctw Lab
Grown Diamonds
LGD Anniversary:
4 LGD244925D 14KY Shared Prong 11 6.5 1 0.0000 Q $1.00 $1.00
Rd. 1/4 Ctw Lab
Grown Diamonds
LGD Anniversary:
5 LGD244933E RG0004161E 14KW Shared Prong 6.5 1 0.0000 Q $1.00 $1.00
11 Rd. 1/3 Ctw Lab
Grown Diamonds
LGD Anniversary:
6 LGD244933D RG0004161E 14KY Shared Prong 11 6.5 1 0.0000 Q $1.00 $1.00
Rd. 1/3 Ctw Lab
Grown Diamonds
LGD Anniversary:
7 LGD244950D RG0004161F 14KY Shared Prong 11 6.5 1 0.0000 Q $1.00 $1.00
Rd. 1/2 Ctw Lab
Grown Diamonds
LGD Anniversary:
8 LGD244950E RG0004161F 14KW Shared Prong 6.5 1 0.0000 Q $1.00 $1.00
11 Rd. 1/2 Ctw Lab
Grown Diamonds
LGD Anniversary:
9 LGD244975E RG0004161G 14KW Shared Prong 6.5 1 0.0000 Q $1.00 $1.00
11 Rd. 3/4 Ctw Lab
Grown Diamonds
LGD Anniversary:
10 LGD244975D RG0004161G 14KY Shared Prong 11 6.5 1 0.0000 Q $1.00 $1.00
Rd. 3/4 Ctw Lab
Grown Diamonds
LGD Anniversary:
11 LGD244901E 14KW Shared Prong 6.5 1 0.0000 Q $1.00 $1.00
11 Rd. 1.00 Ctw Lab
Grown Diamonds
Grand Total: $12.00
RightClick® Copyright © 2026 CFI/Wise Choice Software® All Rights Reserved
20180426.112919[s]; 20180426.112919[s]
oT
redrO
PRESTIGE DIAMONDS PVT. LTD SHEFI DIAMONDS, INC.
BDB BKC ANDHERI E 10 WEST 46TH STREET, 15TH FLOOR
MUMBAI MH INDIA NEW YORK NY 10036 USA
RAVINA/ANKUSH/LOKESH
oT
pihS
SHEFI DIAMONDS, INC. Purchase Order
10 WEST 46TH STREET, 15TH FLOOR
NEW YORK NY 10036 USA
(800) 76-SHEFI (212) 391-1482 Fax: (212) 719-2892
WWW.SHEFIDIAMONDS.COM
INFO@SHEFIDIAMONDS.COM
Order #: 29626
Page #: 2 of 2
P.O. #: STOCK Date: 3/27/2026 Due Date: 3/27/2026 Cancel Date: 3/27/2026
Reference: STOCK Vendor #:P533 Phone #: Ship Via: BRINKS
# Memo # Item # Vendor Item # Job Bag # Description Size Quantity Weight Unit Cost Amount
LGD Anniversary:
12 LGD244901D 14KY Shared Prong 11 6.5 1 0.0000 Q $1.00 $1.00
Rd. 1.00 Ctw Lab
Grown Diamonds
12 0.0000
Grand Total: $12.00
RightClick® Copyright © 2026 CFI/Wise Choice Software® All Rights Reserved
20180426.112919[s]; 20180426.112919[s]
"""
    
    # Initialize data list
    data = []
    
    # Extract using regex
    po_number = re.search(r'PO # (\d+)', extracted_text)
    po_number = po_number.group(1) if po_number else ""
    
    metal_match = re.search(r'SHIMAYRA JEWELLERY ([\w\s]+),', extracted_text)
    metal = metal_match.group(1).strip() if metal_match else ""
    
    desc_match = re.search(r'(\d+\.\d+ CT TW [^\n]+Size-(\d+))', extracted_text)
    if desc_match:
        customer_instruction = desc_match.group(1).replace(f"Size-{desc_match.group(2)}", f"SZ-{desc_match.group(2)}")
        item_size = f"UP{desc_match.group(2).zfill(2)}"
    else:
        customer_instruction = ""
        item_size = ""
    
    style_qty_match = re.search(r'(\d+)\s+([A-Z0-9-]+(?:/[A-Z0-9]+)?)\s+(\d+\.\d+)', extracted_text)
    if style_qty_match:
        sr_no = style_qty_match.group(1)
        style_code = style_qty_match.group(2)
        order_qty = style_qty_match.group(3).split('.')[0]
    else:
        sr_no = ""
        style_code = ""
        order_qty = ""
    
    special_remarks_match = re.search(r'Need Hallmark & Trademark on every piece\.', extracted_text)
    special_remarks = special_remarks_match.group(0) if special_remarks_match else ""

    # Ask for user input for Tone
    tone = input("Enter Tone (Y for Yellow, W for White, P for Pink): ").strip().upper()

    # Determine Metal Code based on Tone + Karat or Platinum
    def determine_metal_code(metal_text, tone_input):
        metal_text = metal_text.upper()
        if "PT" in metal_text or "PLATINUM" in metal_text:
            return "PC95"
        elif "14K" in metal_text:
            return f"G14{tone_input}"
        elif "18K" in metal_text:
            return f"G18{tone_input}"
        else:
            return f"GXX{tone_input}"  # fallback if not found
    
    metal_code = determine_metal_code(metal, tone)

    # Build row data
    item_data = {
        'SrNo': sr_no,
        'StyleCode': style_code.split('-')[0] if '-' in style_code else style_code,  # Only base code like QR0350H
        'ItemSize': item_size,
        'OrderQty': order_qty,
        'OrderItemPcs': 1,
        'Metal': metal_code,
        'Tone': tone,
        'ItemPoNo': po_number,
        'ItemRefNo': "",
        'StockType': "",
        'MakeType': "",
        'CustomerProductionInstruction': customer_instruction,
        'SpecialRemarks': special_remarks,
        'DesignProductionInstruction': "",
        'StampInstruction': "'A' on one side and metal KT on other side of the ring",
        'OrderGroup': "",
        'Certificate': "",
        'SKUNo': "",
        'Basestoneminwt': "",
        'Basestonemaxwt': "",
        'Basemetalminwt': "",
        'Basemetalmaxwt': "",
        'Productiondeliverydate': "",
        'Expecteddeliverydate': "",
        'Blank_Column': "",
        'SetPrice': "",
        'StoneQuality': ""
    }

    data.append(item_data)
    return data


# === Run Processing ===
processed_data = process_po_6393_data()
df = pd.DataFrame(processed_data)

# === Save Output ===
output_filename = 'PO_6393_anayapdf_final.xlsx'
df.to_excel(output_filename, index=False)

# === Display Results ===
print("\n✅ Processed DataFrame:")
print(df.to_string(index=False))
print(f"\n💾 Data successfully saved to '{output_filename}'")



✅ Processed DataFrame:
SrNo StyleCode ItemSize OrderQty  OrderItemPcs Metal Tone ItemPoNo ItemRefNo StockType MakeType CustomerProductionInstruction SpecialRemarks DesignProductionInstruction                                       StampInstruction OrderGroup Certificate SKUNo Basestoneminwt Basestonemaxwt Basemetalminwt Basemetalmaxwt Productiondeliverydate Expecteddeliverydate Blank_Column SetPrice StoneQuality
   5         1                 0             1  GXXY    Y                                                                                                                'A' on one side and metal KT on other side of the ring                                                                                                                                                                        

💾 Data successfully saved to 'PO_6393_anayapdf_final.xlsx'


In [1]:
'''SHEFI NEW PO - ALL COLUMNS CORRECTLY EXTRACTED AND SAVED TO EXCEL'''

import pandas as pd
import re
from datetime import datetime
import pdfplumber

def format_date(date_str):
    """Convert M/D/YYYY to DD-Mon-YY (e.g. 3/27/2026 -> 27-Mar-26)"""
    try:
        return datetime.strptime(date_str.strip(), '%m/%d/%Y').strftime('%d-%b-%y')
    except Exception:
        return date_str.strip()

def extract_shefi_po(pdf_path):
    """
    Extract all PO data page-by-page so each item carries its own correct page number.
    Correctly maps all 17 columns per the specification.
    """
    all_rows = []
    item_counter = 0   # continuous across pages

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ''
            lines = [ln.strip() for ln in page_text.split('\n')]

            # ── Header extraction (per page) ────────────────────────────────
            hdr = {
                'Customer': 'SHEFI DIAMONDS, INC',
                'Order#': '', 'Page#': '', 'PO#': '',
                'Date': '', 'Due Date': '', 'Cancel Date': '',
                'Ref': '', 'Vendor#': '', 'Ship Via': ''
            }

            for line in lines:
                m = re.search(r'Order #:\s*(\d+)', line)
                if m:
                    hdr['Order#'] = m.group(1)

                m = re.search(r'Page #:\s*(\d+\s+of\s+\d+)', line)
                if m:
                    hdr['Page#'] = m.group(1)

                m = re.search(r'P\.O\. #:\s*(\S+)', line)
                if m:
                    hdr['PO#'] = m.group(1)

                # Combined date extraction to avoid mismatching Due/Cancel Date
                m = re.search(
                    r'Date:\s*(\d+/\d+/\d+)\s+Due Date:\s*(\d+/\d+/\d+)\s+Cancel Date:\s*(\d+/\d+/\d+)',
                    line
                )
                if m:
                    hdr['Date']        = format_date(m.group(1))
                    hdr['Due Date']    = format_date(m.group(2))
                    hdr['Cancel Date'] = format_date(m.group(3))

                m = re.search(r'Reference:\s*(\S+)', line)
                if m:
                    hdr['Ref'] = m.group(1)

                m = re.search(r'Vendor #:(\S+)', line)
                if m:
                    hdr['Vendor#'] = m.group(1)

                m = re.search(r'Ship Via:\s*(\S+)', line)
                if m:
                    hdr['Ship Via'] = m.group(1)

            # ── Item extraction (per page) ───────────────────────────────────
            current_item   = None
            current_cat    = None
            expect_item    = False

            # Lines that qualify as category headers (e.g. "LGD Anniversary:")
            # They start with a letter, contain text, and end with ":"
            CAT_PATTERN = re.compile(r'^[A-Za-z][A-Za-z0-9 ]+:$')
            # Keywords that signal a description continuation line
            DESC_KEYWORDS = ('Rd.', 'Ctw', 'Lab', 'Grown', 'Diamond')

            for line in lines:
                # ── Category line ────────────────────────────────────────────
                if CAT_PATTERN.match(line) and not any(
                    kw in line for kw in ('Order', 'Page', 'P.O.', 'Ref', 'Ship', 'Grand', 'Right')
                ):
                    if current_item:
                        # Finalise description and save previous item
                        extras = current_item.pop('_extra', [])
                        current_item['Description'] += (' ' + ' '.join(extras)) if extras else ''
                        all_rows.append(current_item)
                        current_item = None
                    current_cat = line.rstrip(':').strip()
                    expect_item = True
                    continue

                # ── Item data line ────────────────────────────────────────────
                if expect_item:
                    m = re.match(r'^(\d+)\s+([A-Z][A-Z0-9]+)\s+(.+)', line)
                    if m:
                        item_counter += 1
                        item_no = m.group(2)
                        rest    = m.group(3)
                        words   = rest.split()

                        # --- Vendor Item # -----------------------------------
                        # Present when the first word of `rest` is an alphanumeric
                        # product code (not a metal type like 14KW/14KY).
                        # Pattern: starts with 2 uppercase letters, >= 8 chars total.
                        vendor_item = ''
                        w_off = 0
                        if (words
                                and not re.match(r'^14K', words[0])
                                and re.match(r'^[A-Z]{2}', words[0])
                                and len(words[0]) >= 8):
                            vendor_item = words[0]
                            w_off = 1   # metal description starts one word later

                        # --- Metal description (e.g. "14KW Shared Prong") ----
                        metal_desc = ' '.join(words[w_off: w_off + 3]) if len(words) >= w_off + 3 else ''

                        # --- Stone count on same line (e.g. "11" before size) -
                        # Appears as an integer BETWEEN "Prong" and the decimal size.
                        stone_on_line = ''
                        idx_after_metal = w_off + 3
                        if len(words) > idx_after_metal:
                            cand = words[idx_after_metal]
                            if re.match(r'^\d+$', cand):
                                # Verify next word is the decimal size (e.g. 6.5)
                                if (len(words) > idx_after_metal + 1
                                        and re.match(r'^\d+\.\d+$', words[idx_after_metal + 1])):
                                    stone_on_line = cand

                        # --- Size and Quantity --------------------------------
                        # Always: <decimal_size> <qty> 0.0000  (e.g. "6.5 1 0.0000")
                        sq = re.search(r'(\d+\.\d+)\s+(\d+)\s+0\.0000', rest)
                        size_val = sq.group(1) if sq else ''
                        qty_val  = sq.group(2) if sq else ''

                        # --- Build initial description -----------------------
                        desc_parts = [current_cat or '', metal_desc]
                        if stone_on_line:
                            desc_parts.append(stone_on_line)
                        desc = ' '.join(p for p in desc_parts if p)

                        current_item = dict(hdr)
                        current_item.update({
                            '#':             item_counter,
                            'Memo #':        '',
                            'Item #':        item_no,
                            'Vendor Item #': vendor_item,
                            'Description':   desc,
                            'Size':          size_val,
                            'Quantity':      qty_val,
                            '_extra':        []          # temp; removed before saving
                        })
                        expect_item = False
                        continue

                # ── Continuation lines (extra description text) ───────────────
                if current_item and line:
                    if any(kw in line for kw in DESC_KEYWORDS):
                        current_item['_extra'].append(line)

            # Save the last item on this page
            if current_item:
                extras = current_item.pop('_extra', [])
                current_item['Description'] += (' ' + ' '.join(extras)) if extras else ''
                all_rows.append(current_item)

    return all_rows


# ── Run extraction ────────────────────────────────────────────────────────────
pdf_file_path = r'C:\Users\Admin\Desktop\OPS_Tool_10032026\SHEFI_NEW_PO\PDPO#29626.pdf'

rows = extract_shefi_po(pdf_file_path)

COLUMNS = [
    'Customer', 'Order#', 'Page#', 'PO#', 'Date', 'Due Date', 'Cancel Date',
    'Ref', 'Vendor#', 'Ship Via', '#', 'Memo #', 'Item #', 'Vendor Item #',
    'Description', 'Size', 'Quantity'
]

df_shefi = pd.DataFrame(rows)
for col in COLUMNS:
    if col not in df_shefi.columns:
        df_shefi[col] = ''
df_shefi = df_shefi[COLUMNS]

# ── Save to Excel ─────────────────────────────────────────────────────────────
output_path = r'C:\Users\Admin\Desktop\OPS_Tool_10032026\SHEFI_NEW_PO\SHEFI_PO_29626_FINAL.xlsx'
df_shefi.to_excel(output_path, index=False)

# ── Display results ───────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 70)

print(f"Total items extracted : {len(df_shefi)}")
print(f"Output file           : {output_path}")
print("=" * 90)
print(df_shefi.to_string(index=False))

# Verification summary
print("\n" + "=" * 90)
print("VERIFICATION")
print("=" * 90)
for _, row in df_shefi.iterrows():
    print(f"  #{row['#']:>2}  Item#={row['Item #']:<12}  "
          f"VendorItem#={str(row['Vendor Item #']):<12}  "
          f"Page#={row['Page#']:<7}  "
          f"Size={row['Size']:<5}  Qty={row['Quantity']}")
    print(f"       Desc: {row['Description']}")

df_shefi


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


Total items extracted : 12
Output file           : C:\Users\Admin\Desktop\OPS_Tool_10032026\SHEFI_NEW_PO\SHEFI_PO_29626_FINAL.xlsx
           Customer Order#  Page#   PO#      Date  Due Date Cancel Date   Ref Vendor# Ship Via  # Memo #     Item # Vendor Item #                                                          Description Size Quantity
SHEFI DIAMONDS, INC  29626 1 of 2 STOCK 27-Mar-26 27-Mar-26   27-Mar-26 STOCK    P533   BRINKS  1        LGD244910E               LGD Anniversary 14KW Shared Prong 11 Rd. 1/10 Ctw Lab Grown Diamonds  6.5        1
SHEFI DIAMONDS, INC  29626 1 of 2 STOCK 27-Mar-26 27-Mar-26   27-Mar-26 STOCK    P533   BRINKS  2        LGD244910D               LGD Anniversary 14KY Shared Prong 11 Rd. 1/10 Ctw Lab Grown Diamonds  6.5        1
SHEFI DIAMONDS, INC  29626 1 of 2 STOCK 27-Mar-26 27-Mar-26   27-Mar-26 STOCK    P533   BRINKS  3        LGD244925E                LGD Anniversary 14KW Shared Prong 11 Rd. 1/4 Ctw Lab Grown Diamonds  6.5        1
SHEFI DIAMONDS, I

,Customer,Order#,Page#,PO#,Date,Due Date,Cancel Date,Ref,Vendor#,Ship Via,#,Memo #,Item #,Vendor Item #,Description,Size,Quantity
0,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,1,,LGD244910E,,LGD Anniversary 14KW Shared Prong 11 Rd. 1/10 Ctw Lab Grown Diamonds,6.5,1
1,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,2,,LGD244910D,,LGD Anniversary 14KY Shared Prong 11 Rd. 1/10 Ctw Lab Grown Diamonds,6.5,1
2,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,3,,LGD244925E,,LGD Anniversary 14KW Shared Prong 11 Rd. 1/4 Ctw Lab Grown Diamonds,6.5,1
3,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,4,,LGD244925D,,LGD Anniversary 14KY Shared Prong 11 Rd. 1/4 Ctw Lab Grown Diamonds,6.5,1
4,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,5,,LGD244933E,RG0004161E,LGD Anniversary 14KW Shared Prong 11 Rd. 1/3 Ctw Lab Grown Diamonds,6.5,1
5,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,6,,LGD244933D,RG0004161E,LGD Anniversary 14KY Shared Prong 11 Rd. 1/3 Ctw Lab Grown Diamonds,6.5,1
6,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,7,,LGD244950D,RG0004161F,LGD Anniversary 14KY Shared Prong 11 Rd. 1/2 Ctw Lab Grown Diamonds,6.5,1
7,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,8,,LGD244950E,RG0004161F,LGD Anniversary 14KW Shared Prong 11 Rd. 1/2 Ctw Lab Grown Diamonds,6.5,1
8,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,9,,LGD244975E,RG0004161G,LGD Anniversary 14KW Shared Prong 11 Rd. 3/4 Ctw Lab Grown Diamonds,6.5,1
9,"SHEFI DIAMONDS, INC",29626,1 of 2,STOCK,27-Mar-26,27-Mar-26,27-Mar-26,STOCK,P533,BRINKS,10,,LGD244975D,RG0004161G,LGD Anniversary 14KY Shared Prong 11 Rd. 3/4 Ctw Lab Grown Diamonds,6.5,1
